# Jane Street August 2026 puzzle — Andy's Afternoon Amble (my solution)

Andy the ant lives on a truncated tetrahedron: 4 white hexagons (each touching the other 3) and 4 black triangles he avoids. Every afternoon he marks his home hexagon and does a random walk — each step goes to one of the 3 neighboring white hexagons with equal probability — and stops the moment he's back home. He recognizes home by smell but can't tell its edges apart.

Today the ball bounced through a kitchen with an infinite hex-tiled floor: black hexagons each surrounded by 6 white ones, white ones surrounded by alternating black and white. Andy fell off, woke up on a white hexagon of the floor, noticed nothing (locally it looks identical — 3 white neighbors, alternating black), and did his usual amble. He remembers every turn he takes. **Find the exact probability p that by the end of the amble he's figured out he's not on his ball anymore.**

## The idea in one paragraph

The only things Andy ever learns are (1) the sequence of turns he took — back, left, or right at each step — and (2) whether he smells home at each step. On the ball, his position is *completely determined* by the turns. So I can run a "shadow Andy" on the ball using the same turns as the real walk on the floor. Andy gets caught out exactly when the shadow's home-visits don't line up with his real ones. The clean way to make that precise: there's a turn-preserving **covering map** ψ from the floor's white-hexagon graph (a honeycomb lattice) down to the ball's white-hexagon graph (K₄ — four vertices, all adjacent). The shadow's position is just ψ(real position). One useful consequence: whenever real Andy is home, the shadow is home too — so the shadow's first return always happens no later than the real one. Andy stays fooled **iff** his walk gets back home before stepping on any *other* floor hexagon that ψ sends to home. Those "impostor home" hexagons are the fiber ψ⁻¹(home), and

$$p = P(\text{hit the fiber minus home before returning home}).$$

The rest is building ψ explicitly and computing that hitting probability exactly.

## Why the covering map exists

Both graphs are 3-regular and drawn on an orientable surface, so "left" and "right" make sense everywhere. Walks driven by turns are really walks on *darts* (directed edges), pushed around by three moves: B (turn back), L, R. Any turn-word that acts as the identity on the honeycomb's darts is built from three elementary relations: B² = id (back twice), L⁶ = id (six lefts loop a hexagonal face), and (LB)³ = id (spinning around a degree-3 vertex). On K₄ drawn on the sphere, faces are the 3-cycles around the black triangles, so L³ = id there — which means all three honeycomb relations hold on K₄ too (L⁶ = (L³)² = id). Relations only need to hold one way for a quotient map, so the equivariant map on darts exists: pick where one dart goes, and everything else is forced. Geometrically it's a branched covering — each floor hexagon face wraps twice around a ball triangle — but the walk never sees face interiors, so the branching is invisible to Andy. Below I build the map by brute force and let the computer check it's consistent.

In [1]:
from collections import deque
import numpy as np

# ---------- the ball: K4 with a rotation system (cyclic order of neighbors at each vertex) ----------
def faces_are_triangles(rho):
    darts = [(u,v) for u in range(4) for v in range(4) if u != v]
    def L(d):
        u,v = d
        nb = rho[v]; i = nb.index(u)
        return (v, nb[(i+1)%3])
    for d in darts:
        x = d
        for _ in range(3): x = L(x)
        if x != d: return False
    return True

rho = {0:[1,2,3], 1:[0,3,2], 2:[0,1,3], 3:[0,2,1]}   # an oriented tetrahedron
assert faces_are_triangles(rho)

def k4_move(d, t):
    u,v = d
    if t == 'B': return (v,u)
    nb = rho[v]; i = nb.index(u)
    return (v, nb[(i+1)%3]) if t == 'L' else (v, nb[(i-1)%3])

# quick relation checks on the ball: B^2, L^3, (LB)^3 all identity
darts = [(u,v) for u in range(4) for v in range(4) if u != v]
def apply_word(d, word, mover):
    for t in word: d = mover(d, t)
    return d
assert all(apply_word(d, 'BB', k4_move) == d for d in darts)
assert all(apply_word(d, 'LLL', k4_move) == d for d in darts)
assert all(apply_word(d, 'LBLBLB', k4_move) == d for d in darts)
print("ball checks pass: B^2 = L^3 = (LB)^3 = id on all 12 darts")

ball checks pass: B^2 = L^3 = (LB)^3 = id on all 12 darts


In [2]:
# ---------- the floor: white hexagons of the 3-colored hex tiling ----------
# axial coordinates (q, r); color = (q - r) mod 3; color 0 = black, colors 1 & 2 = white.
# Every hexagon's 6 neighbors alternate +1 / -1 in color, which matches the puzzle picture:
# whites see alternating black/white, blacks see 6 whites.
def color(v): return (v[0] - v[1]) % 3

W1 = [(1,0), (0,-1), (-1,1)]    # CCW order of white neighbors of a color-1 hex
W2 = [(1,-1), (-1,0), (0,1)]    # CCW order of white neighbors of a color-2 hex
def wnbrs(v):
    ds = W1 if color(v) == 1 else W2
    return [(v[0]+d[0], v[1]+d[1]) for d in ds]

def hc_move(d, t):
    u,v = d
    if t == 'B': return (v,u)
    nb = wnbrs(v); i = nb.index(u)
    return (v, nb[(i+1)%3]) if t == 'L' else (v, nb[(i-1)%3])

home = (1, 0)
d0 = (wnbrs(home)[0], home)
assert apply_word(d0, 'LLLLLL', hc_move) == d0     # floor faces are 6-cycles
assert apply_word(d0, 'LBLBLB', hc_move) == d0
print("floor checks pass: faces are 6-cycles, (LB)^3 = id")

floor checks pass: faces are 6-cycles, (LB)^3 = id


In [3]:
# ---------- build the covering psi by BFS on darts ----------
# send one dart into home to a dart into ball-vertex 0, then propagate with B/L/R.
R = 30
def inpatch(v): return abs(v[0]) <= R and abs(v[1]) <= R and abs(v[0]+v[1]) <= R

phi = {d0: (1, 0)}          # dart-level map; so psi(home) = 0
Q = deque([d0]); conflicts = 0
while Q:
    d = Q.popleft()
    for t in 'BLR':
        nd = hc_move(d, t)
        if not (inpatch(nd[0]) and inpatch(nd[1])): continue
        img = k4_move(phi[d], t)
        if nd in phi:
            if phi[nd] != img: conflicts += 1
        else:
            phi[nd] = img; Q.append(nd)

psi = {}
for (u,v), (a,b) in phi.items():
    assert psi.setdefault(v, b) == b     # well-defined on vertices
assert conflicts == 0
print(f"covering built: {len(phi)} darts, {len(psi)} vertices, 0 conflicts")

# the "rainbow" condition = definition of a covering onto K4:
# every floor hexagon sees the three OTHER ball-labels among its neighbors, once each
for v, l in psi.items():
    ns = wnbrs(v)
    if all(n in psi for n in ns):
        assert sorted(psi[n] for n in ns) == sorted(set(range(4)) - {l})
print("rainbow condition holds everywhere")

fiber = sorted(v for v, l in psi.items() if l == 0)
print(f"fiber over home's label: {len(fiber)} of {len(psi)} whites "
      f"= {len(fiber)/len(psi):.4f}  (should be about 1/4)")

covering built: 5460 darts, 1860 vertices, 0 conflicts
rainbow condition holds everywhere
fiber over home's label: 460 of 1860 whites = 0.2473  (should be about 1/4)


In [4]:
# a look at the label pattern near home ('.' = black hexagon). home is the 0 at (q,r)=(1,0).
print("      q = -4 -3 -2 -1  0  1  2  3  4  5")
for r in range(4, -5, -1):
    row = []
    for q in range(-4, 6):
        v = (q, r)
        row.append('.' if color(v) == 0 else str(psi.get(v, '?')))
    print(f"r = {r:2d}:    " + '  '.join(row))

      q = -4 -3 -2 -1  0  1  2  3  4  5
r =  4:    1  0  .  0  1  .  1  0  .  0
r =  3:    3  .  3  2  .  2  3  .  3  2
r =  2:    .  0  1  .  1  0  .  0  1  .
r =  1:    3  2  .  2  3  .  3  2  .  2
r =  0:    1  .  1  0  .  0  1  .  1  0
r = -1:    .  2  3  .  3  2  .  2  3  .
r = -2:    1  0  .  0  1  .  1  0  .  0
r = -3:    3  .  3  2  .  2  3  .  3  2
r = -4:    .  0  1  .  1  0  .  0  1  .


The pattern is periodic, and label-0 hexagons (home's impostors) make up exactly a quarter of the whites. The nearest impostors sit at graph-distance 3 from home: they're the hexagons *directly across* each of home's three adjacent black hexagons. That makes sense — turning left twice (or right twice) closes a triangle on the ball, so the shadow is back home, but on the floor it only gets you halfway around a black hexagon.

Quick sanity check against the example in the puzzle: a 2-step amble (out and straight back, the turn "B") returns home on both surfaces, so those afternoons — exactly 1/3 of them — are never discoveries. So we should end up with p ≤ 2/3.

## Monte Carlo first

Before doing anything exact, I just run the coupled walk a couple million times: real Andy on the floor, shadow Andy on the ball, same random turns, and check whether the first home-visit of either one is a simultaneous home-visit of both.

In [5]:
import random
random.seed(7)

N = 2_000_000
discoveries = 0
for _ in range(N):
    k = random.randrange(3)
    pd = (home, wnbrs(home)[k])          # floor dart (from, to)
    sd = (0, rho[0][k])                  # ball dart, aligned arbitrarily (symmetry: doesn't matter)
    while True:
        shadow_home = (sd[1] == 0)
        real_home = (pd[1] == home)
        if shadow_home or real_home:
            if not (shadow_home and real_home): discoveries += 1
            break
        t = random.choice('BLR')
        u, v = pd
        nb = wnbrs(v); i = nb.index(u)
        pd = (v, u) if t == 'B' else ((v, nb[(i+1)%3]) if t == 'L' else (v, nb[(i-1)%3]))
        sd = k4_move(sd, t)

print(f"Monte Carlo estimate: p ≈ {discoveries/N:.4f}")

Monte Carlo estimate: p ≈ 0.5493


## Exact computation

The fiber is periodic, so I fold the whole problem onto a finite torus. First find the lattice of translations that preserve the labeling — it turns out to be generated by (2,2) and (−4,2) in axial coordinates, giving a fundamental cell of 12 hexagons (8 white, of which 2 carry label 0: home's class and the impostor class).

The walk on the floor is then a walk on this 8-state torus plus a **winding number** in ℤ² recording which copy of the cell you're in. "Return home before hitting an impostor" means: first label-0 visit lands in home's class *with winding exactly zero*. Fourier handles the winding: give every edge-crossing a phase $e^{i\theta\cdot w}$, solve a tiny 6×6 absorbing linear system for each θ, and average over θ ∈ [0,2π)². The integrand is analytic and periodic, so a plain grid average converges spectrally — if the digits freeze as the grid grows, they're exact for all practical purposes.

In [6]:
# translation lattice preserving psi
def preserves(t):
    seen = 0
    for v, l in psi.items():
        w = (v[0]+t[0], v[1]+t[1])
        if w in psi:
            seen += 1
            if psi[w] != l: return False
    return seen > 200

small = [t for t in [(a,b) for a in range(-6,7) for b in range(-6,7)]
         if t != (0,0) and (t[0]-t[1]) % 3 == 0 and preserves(t)]
small.sort(key=lambda t: (abs(t[0])+abs(t[1]), t))
print("smallest label-preserving translations:", small[:4])

t1, t2 = (2,2), (-4,2)
M = np.array([[2,-4],[2,2]], float); Minv = np.linalg.inv(M)

def canon(v):
    """reduce v modulo the lattice; return (representative, winding in Z^2)"""
    ab = Minv @ np.array(v, float)
    a, b = int(np.floor(ab[0]+1e-9)), int(np.floor(ab[1]+1e-9))
    return (v[0]-2*a+4*b, v[1]-2*a-2*b), (a, b)

reps = sorted({canon((q,r))[0] for q in range(-6,7) for r in range(-6,7) if color((q,r)) != 0})
assert len(reps) == 8
labels = {v: psi[v] for v in reps}
Sbar = [v for v in reps if labels[v] == 0]
obar = canon(home)[0]
print("torus states and ball-labels:", labels)
print("absorbing (label-0) states:", Sbar, "   home's class:", obar)

smallest label-preserving translations: [(-2, -2), (2, 2), (-6, 0), (-4, 2)]
torus states and ball-labels: {(-3, 2): 0, (-2, 2): 1, (-2, 3): 3, (-1, 1): 2, (-1, 3): 2, (0, 1): 3, (0, 2): 1, (1, 2): 0}
absorbing (label-0) states: [(-3, 2), (1, 2)]    home's class: (-3, 2)


In [7]:
trans = [v for v in reps if labels[v] != 0]
idx = {v: i for i, v in enumerate(trans)}
moves = {v: [canon(u) for u in wnbrs(v)] for v in reps}

def f_theta(th1, th2):
    """E[ e^{i theta . winding} ; first label-0 hit is home's class ], starting one step out of home"""
    A = np.eye(len(trans), dtype=complex)
    b = np.zeros(len(trans), dtype=complex)
    for v in trans:
        i = idx[v]
        for (u, w) in moves[v]:
            ph = np.exp(1j*(w[0]*th1 + w[1]*th2)) / 3.0
            if u in idx: A[i, idx[u]] -= ph
            elif u == obar: b[i] += ph
    g = np.linalg.solve(A, b)
    val = 0
    for (u, w) in moves[obar]:
        ph = np.exp(1j*(w[0]*th1 + w[1]*th2)) / 3.0
        val += ph * (g[idx[u]] if u in idx else (1.0 if u == obar else 0.0))
    return val

for N in (32, 64, 128, 256):
    ths = 2*np.pi*np.arange(N)/N
    q = np.mean([f_theta(a, b) for a in ths for b in ths]).real
    print(f"grid {N}x{N}:   q = {q:.15f}   ->   p = {1-q:.15f}")

grid 32x32:   q = 0.450000000000000   ->   p = 0.550000000000000
grid 64x64:   q = 0.450000000000000   ->   p = 0.550000000000000


grid 128x128:   q = 0.450000000000000   ->   p = 0.550000000000000


grid 256x256:   q = 0.450000000000000   ->   p = 0.550000000000000


q = 9/20 on the nose, at every grid size, to 14 digits. So p = 11/20.

## Double-checking with a totally different method

A rational answer for an infinite-lattice problem deserves suspicion, so here's an independent computation: solve the hitting problem directly as a big sparse linear system on patches of increasing radius, once treating the unknown boundary as "discovered" (upper bound on p) and once as "returned home" (lower bound). Because the absorbing set has density 1/4, the walk essentially never reaches the boundary, and the two bounds should pinch together fast — they bracket the true answer rigorously.

In [8]:
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import spsolve

def label(v): return psi[canon(v)[0]]

for Rp in (20, 40, 60):
    verts = [(q,r) for q in range(-Rp,Rp+1) for r in range(-Rp,Rp+1)
             if abs(q+r) <= Rp and color((q,r)) != 0]
    vset = set(verts)
    tr = [v for v in verts if label(v) != 0]
    tix = {v: i for i, v in enumerate(tr)}
    for bc, name in ((1.0, 'upper'), (0.0, 'lower')):
        A = lil_matrix((len(tr), len(tr))); b = np.zeros(len(tr))
        for v in tr:
            i = tix[v]; A[i,i] = 1.0
            for u in wnbrs(v):
                if u not in vset:      b[i] += bc/3.0
                elif label(u) == 0:
                    if u != home:      b[i] += 1.0/3.0   # impostor: discovered
                else:                  A[i, tix[u]] -= 1.0/3.0
        h = spsolve(csr_matrix(A), b)
        p = sum(h[tix[u]] if label(u) != 0 else (0.0 if u == home else 1.0)
                for u in wnbrs(home)) / 3.0
        print(f"radius {Rp} ({name} bound):  p = {p:.14f}")

print()
print("p = 11/20 =", 11/20)

radius 20 (upper bound):  p = 0.55000000000000
radius 20 (lower bound):  p = 0.55000000000000
radius 40 (upper bound):  p = 0.55000000000000
radius 40 (lower bound):  p = 0.55000000000000


radius 60 (upper bound):  p = 0.55000000000000
radius 60 (lower bound):  p = 0.55000000000000

p = 11/20 = 0.55


## Wrapping up

Three methods, one answer:

- **Monte Carlo** (2M coupled walks): p ≈ 0.549
- **Torus + Fourier winding integral**: q = 9/20 exactly, so p = 11/20, stable to 14 digits across grid sizes
- **Direct sparse solves with bracketing boundary conditions**: upper and lower bounds both equal 0.55000000000000

The story: Andy's turn memory turns his ball into a shadow world sitting under the kitchen floor via a covering map. A quarter of all white floor hexagons are "impostor homes" — places where the shadow swears he's home but there's no pheromone smell (the closest ones are just across each black hexagon touching home, 3 steps away). Andy stays happily oblivious only if his amble closes up before touching any impostor, which happens with probability 9/20. So the probability he discovers the truth is

$$p = \boxed{\dfrac{11}{20}}$$